Competition Link: https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data

The training data, comprising time series of features store_nbr, family, and onpromotion as well as the target sales.
1. `store_nbr` identifies the store at which the products are sold.
2. `family` identifies the type of product sold.
3. `sales` gives the total sales for a product family at a particular store at a given date. Fractional values are possible since products can be sold in fractional units (1.5 kg of cheese, for instance, as opposed to 1 bag of chips).
5. `onpromotion` gives the total number of items in a product family that were being promoted at a store at a given date

metrics: RMSLE

Import modules

In [1]:
import kaggle
import pandas as pd
import polars as pl
import numpy as np
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from prophet import Prophet

/home/bareck/anaconda3/envs/ds_projects/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Seed set

In [2]:
np.random.seed = 66

Uncomment the cell below if you don't have the datasets yet

In [ ]:
# !kaggle competitions download -c store-sales-time-series-forecasting
# !unzip store-sales-time-series-forecasting.zip

In [3]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
oil_df = pd.read_csv('oil.csv')
transactions_df = pd.read_csv('transactions.csv')
stores_df = pd.read_csv('stores.csv')
hoildays_df = pd.read_csv('holidays_events.csv')

Dataset quick walkthrough

In [16]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[ns]
 2   store_nbr    int64         
 3   family       object        
 4   sales        float64       
 5   onpromotion  int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(1)
memory usage: 137.4+ MB


In [ ]:
train_df['family'].unique()

In [ ]:
train_df['family'].nunique()

In [4]:
train_df.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [15]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           28512 non-null  int64         
 1   date         28512 non-null  datetime64[ns]
 2   store_nbr    28512 non-null  int64         
 3   family       28512 non-null  object        
 4   onpromotion  28512 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 1.1+ MB


In [ ]:
oil_df.info()

In [ ]:
print(oil_df['date'].min())

In [ ]:
print(oil_df['date'].max())

In [ ]:
transactions_df.info()

In [ ]:
print(transactions_df['date'].min())

In [ ]:
print(transactions_df['date'].max())

In [ ]:
stores_df.info()

In [ ]:
city_counts = stores_df['city'].value_counts()

In [ ]:
plt.figure(figsize=(8,6))
stores_df['city'].value_counts().plot(kind='bar')
plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True)) # Make y-axis to be integer only
plt.xticks(rotation=90)
plt.title('Proportion of stores by city')
plt.show()

In [ ]:
plt.pie(stores_df['state'].value_counts(), 
        labels=stores_df['state'].value_counts().index,
        autopct="%1.1f%%")
plt.title('Proportion of Stores by States')
plt.show()

In [ ]:
hoildays_df.info()

Feature Engineering

1. aggregation
2. datetime format
3. lag
4. Merge (optional/ considering)

datetime format()

In [6]:
train_df['date'] = pd.to_datetime(train_df['date'], format='%Y-%m-%d')
test_df['date'] = pd.to_datetime(train_df['date'], format='%Y-%m-%d')
oil_df['date'] = pd.to_datetime(oil_df['date'], format='%Y-%m-%d')
hoildays_df['date'] = pd.to_datetime(hoildays_df['date'], format='%Y-%m-%d')
transactions_df['date'] = pd.to_datetime(transactions_df['date'], format='%Y-%m-%d')

Aggregation

Combine train dataset with test dataset

In [7]:
df = pd.concat([train_df, test_df], axis=0)

In [14]:
df[df['sales'].isna()].count()

id             28512
date           28512
store_nbr      28512
family         28512
sales              0
onpromotion    28512
dtype: int64

Merge

In [17]:
df = df.merge(oil_df, on='date', how='left')
df = df.merge(hoildays_df, on='date', how='left')
df = df.merge(transactions_df, on=['date', 'store_nbr'], how='left')
df = df.merge(stores_df, on='store_nbr', how='left')

In [20]:
df = df.set_index(['store_nbr', 'date', 'family'])

idx

In [24]:
idx = pd.IndexSlice

In [28]:
df.loc[idx[:,:, ['AUTOMOTIVE', 'BOOKS']]]

,,,id,sales,onpromotion,dcoilwtico,type_x,locale,locale_name,description,transferred,transactions,city,state,type_y,cluster
store_nbr,date,family,,,,,,,,,,,,,,
1,2013-01-01,AUTOMOTIVE,0,0.0,0,NaN,Holiday,National,Ecuador,Primer dia del ano,False,NaN,Quito,Pichincha,D,13
10,2013-01-01,AUTOMOTIVE,33,0.0,0,NaN,Holiday,National,Ecuador,Primer dia del ano,False,NaN,Quito,Pichincha,C,15
11,2013-01-01,AUTOMOTIVE,66,0.0,0,NaN,Holiday,National,Ecuador,Primer dia del ano,False,NaN,Cayambe,Pichincha,B,6
12,2013-01-01,AUTOMOTIVE,99,0.0,0,NaN,Holiday,National,Ecuador,Primer dia del ano,False,NaN,Latacunga,Cotopaxi,C,15
13,2013-01-01,AUTOMOTIVE,132,0.0,0,NaN,Holiday,National,Ecuador,Primer dia del ano,False,NaN,Latacunga,Cotopaxi,C,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54,2013-01-16,BOOKS,3029239,NaN,0,94.28,NaN,NaN,NaN,NaN,NaN,686.0,El Carmen,Manabi,C,3
6,2013-01-16,BOOKS,3029272,NaN,0,94.28,NaN,NaN,NaN,NaN,NaN,1677.0,Quito,Pichincha,D,13
7,2013-01-16,BOOKS,3029305,NaN,0,94.28,NaN,NaN,NaN,NaN,NaN,1619.0,Quito,Pichincha,D,8


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 3082860 entries, (1, Timestamp('2013-01-01 00:00:00'), 'AUTOMOTIVE') to (9, Timestamp('2013-01-16 00:00:00'), 'SEAFOOD')
Data columns (total 14 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            int64  
 1   sales         float64
 2   onpromotion   int64  
 3   dcoilwtico    float64
 4   type_x        object 
 5   locale        object 
 6   locale_name   object 
 7   description   object 
 8   transferred   object 
 9   transactions  float64
 10  city          object 
 11  state         object 
 12  type_y        object 
 13  cluster       int64  
dtypes: float64(3), int64(3), object(8)
memory usage: 341.1+ MB
